# Extra · entrenar sin que explote la VRAM

> **No es el eje del módulo.** Los mismos tres trucos reaparecen en `sft_lora_noticias.py` (batch efectivo, AMP, a veces checkpoint). Apréndelos con el juguete.

## Dummy · tres palancas, un solo objetivo

El T4 de Colab no aguanta el batch que el paper usó. No estás condenado. Hay tres palancas que **no cambian la matemática del descenso**, solo cómo lo pagas en memoria:

1. **Acumulación de gradiente.** Ves 16 ejemplos, no actualizas pesos, ves otros 16, *entonces* actualizas. Es como un batch de 32 que no cupo.
2. **AMP (mixed precision).** En GPU, las cuentas gordas van en 16 bits; los pesos sensibles se actualizan con un scaler. En CPU casi no paga.
3. **Activation checkpointing.** No guardas todas las activaciones del forward. Las recalculas en el backward. Menos RAM, más tiempo. En CPU se ve *más lento*; es el punto.

## Matemática

**Batch efectivo.** Si $B_{\mathrm{micro}}$ es lo que cabe y $G$ los pasos que acumulas,

$$
B_{\mathrm{eff}}=B_{\mathrm{micro}}\times G,\qquad
\hat g = \frac{1}{G}\sum_{g=1}^{G}\hat g^{(g)}_{\mathrm{micro}}.
$$

(En el código: `loss = loss / accum_steps` *antes* de `backward`, para que la escala coincida con un batch $B_{\mathrm{eff}}$.)

**AMP.** Se calcula $Wx$ en fp16/bf16 y se escala la pérdida para que los gradientes chicos no se hagan underflow a $0$. El `GradScaler` es ese volumen.

**Checkpointing.** Sea $F=f_L\circ\cdots\circ f_1$ el stack de bloques. En vez de guardar todas las activaciones $a_\ell$, guardas de vez en cuando y rearmas $a_\ell=f_\ell(a_{\ell-1})$ en el backward. Intercambias FLOPs por bytes.

## Qué comparar en el resumen

Tres corridas, **mismo** $B_{\mathrm{eff}}=32$:

| # | batch físico | accum | checkpoint | qué deberías ver |
|---|---|---|---|---|
| 1 | 32 | 1 | no | baseline |
| 2 | 16 | 2 | no | val_loss parecida, a veces más lenta |
| 3 | 16 | 2 | sí | menos `peak_mem` en CUDA; en CPU, más `time_s` |

Si 1 y 2 dan val_loss *muy* distinta, hay un bug de escala (olvidaste dividir el loss). Si 3 explota, `checkpoint` necesita `use_reentrant=False` en PyTorch nuevo: anótalo, no es tu modelo.


In [ ]:
# Colab: descomenta las dos líneas de abajo (una vez).
# %pip install -q torch
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/quijote.txt

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import List, Optional, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEXT_PATH = "quijote.txt"
print(f"device={DEVICE}  archivo={TEXT_PATH}")

import time

try:
    from torch.utils.checkpoint import checkpoint
except Exception:
    checkpoint = None


In [ ]:
def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read().replace("\r\n", "\n").replace("\r", "\n")


def build_vocab(text: str):
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos


def encode(text: str, stoi: dict) -> torch.Tensor:
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long)


def decode(ids: List[int], itos: dict) -> str:
    return "".join(itos[i] for i in ids)


In [ ]:
class CharWindowDataset(Dataset):
    """Ventanas x → y corrido +1. Si max_windows no es None, muestrea inicios al azar.

    Dummy: el Quijote tiene ~2M letras. Recorrer TODAS las ventanas en CPU no cabe
    en una sesión. Muestrear M inicios es el mismo examen, con menos filas.
    """

    def __init__(self, data_ids: torch.Tensor, seq_len: int, max_windows: Optional[int] = 8000, seed: int = 0):
        self.data = data_ids
        self.seq_len = seq_len
        n = max(0, len(self.data) - self.seq_len - 1)
        if max_windows is not None and n > max_windows:
            g = torch.Generator().manual_seed(seed)
            self.index = torch.randint(0, n, (max_windows,), generator=g)
        else:
            self.index = torch.arange(n)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i: int):
        idx = int(self.index[i])
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y


El `MiniTransformerLM` de abajo acepta `use_checkpointing`. Cada bloque se recomputea en el backward. El resto (atención causal, FFN, posición) es el del ejercicio 3.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, 1, T, T)
        att = att.masked_fill(mask == 0, float("-inf"))
        att = torch.softmax(att, dim=-1)
        att = self.dropout(att)

        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.dropout(self.proj(out))


class MLP(nn.Module):
    def __init__(self, n_embd: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(n_embd, 4 * n_embd)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(self.act(self.fc1(x))))


class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class MiniTransformerLM(nn.Module):
    def __init__(self, vocab_size: int, seq_len: int, n_embd: int = 128, n_head: int = 4, n_layer: int = 2, dropout: float = 0.1, use_checkpointing: bool = False):
        super().__init__()
        self.seq_len = seq_len
        self.use_checkpointing = use_checkpointing and (checkpoint is not None)
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(seq_len, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.size()
        assert T <= self.seq_len, "T excede seq_len del modelo"
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = self.drop(tok + pos.unsqueeze(0))
        for blk in self.blocks:
            x = checkpoint(blk, x) if self.use_checkpointing else blk(x)
        return self.head(self.ln_f(x))


In [ ]:
@dataclass
class TrainConfig:
    seq_len: int = 64
    batch_size: int = 32
    accum_steps: int = 1
    n_embd: int = 128
    n_head: int = 4
    n_layer: int = 2
    dropout: float = 0.1
    lr: float = 3e-4
    epochs: int = 2
    max_windows: int = 2500
    device: str = DEVICE
    use_amp: bool = False
    use_checkpointing: bool = False


def train_one_epoch(model, loader, optimizer, device, use_amp: bool, accum_steps: int):
    model.train()
    crit = nn.CrossEntropyLoss()
    total = 0.0
    n = 0
    use_amp = use_amp and device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    optimizer.zero_grad(set_to_none=True)
    for step, (x, y) in enumerate(loader, start=1):
        x, y = x.to(device), y.to(device)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(x)
            B, T, V = logits.shape
            loss = crit(logits.view(B * T, V), y.view(B * T)) / accum_steps
        scaler.scale(loss).backward()
        if step % accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        total += loss.item() * accum_steps
        n += 1
    return total / max(1, n)


@torch.no_grad()
def eval_loss(model, loader, device):
    model.eval()
    crit = nn.CrossEntropyLoss()
    total = 0.0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        B, T, V = logits.shape
        loss = crit(logits.view(B * T, V), y.view(B * T))
        total += loss.item()
        n += 1
    return total / max(1, n)


def run_experiment(text_path: str, cfg: TrainConfig):
    text = load_text(text_path)
    stoi, _ = build_vocab(text)
    ids = encode(text, stoi)
    split = int(0.9 * len(ids))
    train_ds = CharWindowDataset(ids[:split], cfg.seq_len, cfg.max_windows, seed=0)
    val_ds = CharWindowDataset(ids[split:], cfg.seq_len, min(600, cfg.max_windows), seed=1)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=True)
    device = torch.device(cfg.device)
    model = MiniTransformerLM(
        len(stoi), cfg.seq_len, cfg.n_embd, cfg.n_head, cfg.n_layer, cfg.dropout,
        use_checkpointing=cfg.use_checkpointing,
    ).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    vl = ppl = float("nan")
    for epoch in range(1, cfg.epochs + 1):
        tr = train_one_epoch(model, train_loader, optim, device, cfg.use_amp, cfg.accum_steps)
        vl = eval_loss(model, val_loader, device)
        ppl = math.exp(vl) if vl < 20 else float("inf")
        print(f"Epoch {epoch:02d} | train_loss={tr:.4f} | val_loss={vl:.4f} | val_ppl={ppl:.2f}")
    peak = torch.cuda.max_memory_allocated() / 1024**2 if device.type == "cuda" else None
    return {"val_loss": vl, "val_ppl": ppl, "time_s": round(time.time() - t0, 2), "peak_mem_mib": round(peak, 2) if peak else None, "cfg": cfg}


text_path = TEXT_PATH
experiments = [
    TrainConfig(batch_size=32, accum_steps=1, use_checkpointing=False),
    TrainConfig(batch_size=16, accum_steps=2, use_checkpointing=False),
    TrainConfig(batch_size=16, accum_steps=2, use_checkpointing=True),
]
results = []
for i, cfg in enumerate(experiments, start=1):
    print("\n" + "=" * 80)
    print(f"Exp {i}: batch={cfg.batch_size} accum={cfg.accum_steps} ckpt={cfg.use_checkpointing} device={cfg.device}")
    results.append(run_experiment(text_path, cfg))

print("\nRESUMEN (mismo B_eff = 32)")
for r in results:
    c = r["cfg"]
    print({
        "batch": c.batch_size, "accum": c.accum_steps, "ckpt": c.use_checkpointing,
        "val_loss": r["val_loss"], "time_s": r["time_s"], "peak_mem_mib": r["peak_mem_mib"],
    })
